# Customer Churn Model Training Experiment

Notebook thực hiện các bước: đọc dữ liệu, tiền xử lý, train/test split, huấn luyện mô hình, đánh giá và chọn mô hình có F1 score tốt nhất.

In [8]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    import xgboost as xgb

# Notebook nằm trong notebooks/, còn mã nguồn và data ở thư mục project.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from prepare_data import main as prepare_dataset

PROCESSED_PATH = ROOT / 'data' / 'processed' / 'Customer-Churn-processed.csv'
if not PROCESSED_PATH.exists():
    prepare_dataset()

df = pd.read_csv(PROCESSED_PATH)
print('Dataset shape:', df.shape)
print(df.head().to_string(index=False))
print('\nTarget distribution:\n', df['Churn'].value_counts().sort_index().to_string())

ModuleNotFoundError: No module named 'prepare_data'

In [ ]:
# 1) Tiền xử lý và split train/test
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train distribution: {y_train.value_counts().sort_index().to_dict()}')
print(f'y_test distribution: {y_test.value_counts().sort_index().to_dict()}')

In [ ]:
# 2) Khởi tạo mô hình
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        class_weight='balanced'
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

for name, model in models.items():
    print(f'\n=== Training {name} ===')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    print('Confusion Matrix:
', cm)
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')
    print(f'ROC-AUC: {roc_auc:.4f}')

In [ ]:
# 3) So sánh mô hình theo F1 score
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    result = {
        'Model': name,
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, y_prob),
        'ConfusionMatrix': confusion_matrix(y_test, y_pred)
    }
    results.append(result)

results_df = pd.DataFrame(results).sort_values(by='F1', ascending=False).reset_index(drop=True)
print('\n=== Summary Comparison ===')
print(results_df.round(4).to_string(index=False))

best = results_df.iloc[0]
print(f'\nModel được chọn theo F1 tốt nhất: {best[Model]}')
print(f'F1 Score: {best[F1]:.4f}')
print(f'Precision: {best[Precision]:.4f}')
print(f'Recall: {best[Recall]:.4f}')
print(f'ROC-AUC: {best[ROC_AUC]:.4f}')